# EuroSAT · Phân loại ảnh vệ tinh khi ít nhãn (macro-F1) - KHÔNG dùng trọng số pretrained

Luật thi: chỉ dùng kiến trúc mô hình, **trọng số phải tự huấn luyện** trên dữ liệu được cấp. Notebook chia 3 tầng:

| Tầng | Cách làm | Thời gian | Kết quả |
|---|---|---|---|
| **1** | Đặc trưng thủ công (màu, histogram, LBP bất biến khi xoay, GLCM, gradient) trên ảnh train + bản làm hỏng, rồi LogisticRegression / XGBoost | ~2-3 phút (CPU) | `submission.csv` ngay |
| **2a** | **ResNet-18 khởi tạo ngẫu nhiên** (`weights=None`, chỉnh cho ảnh 64×64), train từ đầu với augmentation mô phỏng test; số epoch tự tính theo ngân sách | `BUDGET_FT` phút | ghi đè nếu val hard tốt hơn |
| **2b** | Pseudo-label ảnh `unlabeled` (không dùng ảnh test), train lại trên train + val + pseudo, rồi ensemble | `BUDGET_PS` phút | `submission.csv` cuối |

**Ý tưởng chính**: ảnh test bị xoay/lật, nhiễu mạnh, mờ, lệch màu, JPEG chất lượng rất thấp, nên huấn luyện bằng **augmentation mô phỏng đúng các biến đổi đó**,
đánh giá trên **val bị làm hỏng** ("val hard") chứ không chỉ val sạch, và dự đoán với **TTA 8 phép xoay/lật**.

Thư viện: `torch`, `torchvision` (chỉ kiến trúc), `numpy`, `pandas`, `Pillow`, `scikit-image`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn`, `tqdm`.

In [ ]:
import os, io, random, time
import torch          # import trước pandas để tránh lỗi DLL trên Windows
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms as T
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

DATA = '.'          # thư mục chứa train/, val/, unlabeled/, test/
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
BS = 128
BUDGET_FT = 12      # phút cho tầng 2a
BUDGET_PS = 12      # phút cho tầng 2b
T_START = time.time()
open('train_log.txt', 'w').close()


def log(*a):
    """In ra màn hình và ghi vào train_log.txt (xem được tiến độ khi chạy bằng nbconvert)."""
    msg = f'[{(time.time() - T_START) / 60:5.1f} phút] ' + ' '.join(map(str, a)); print(msg)
    with open('train_log.txt', 'a', encoding='utf-8') as f: f.write(msg + '\n')


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


log(torch.__version__, DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

## 1. Đọc dữ liệu

In [ ]:
classes = pd.read_csv(f'{DATA}/classes.csv')['label'].tolist()
c2i = {c: i for i, c in enumerate(classes)}


def load_imgs(split, ids, ext):
    return [Image.open(f'{DATA}/{split}/{i}.{ext}').convert('RGB') for i in tqdm(ids, desc=split)]


tr = pd.read_csv(f'{DATA}/train_labels.csv')
va = pd.read_csv(f'{DATA}/val_labels.csv')
unl_ids = sorted(f[:-4] for f in os.listdir(f'{DATA}/unlabeled'))
test_ids = sorted(f[:-4] for f in os.listdir(f'{DATA}/test'))

Xtr, ytr = load_imgs('train', tr.id, 'jpg'), tr.label.map(c2i).values
Xva, yva = load_imgs('val', va.id, 'jpg'), va.label.map(c2i).values
Xunl = load_imgs('unlabeled', unl_ids, 'jpg')
Xte = load_imgs('test', test_ids, 'png')
log('train/val/unlabeled/test:', len(Xtr), len(Xva), len(Xunl), len(Xte))

## 2. EDA: ảnh train (sạch) vs ảnh test (bị biến đổi)

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for k in range(8):
    ax[0, k].imshow(Xtr[k]); ax[0, k].set_title(classes[ytr[k]], fontsize=8)
    ax[1, k].imshow(Xte[k]); ax[1, k].set_title(test_ids[k], fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.suptitle('hàng trên: train (sạch) · hàng dưới: test (bị biến đổi)'); plt.tight_layout(); plt.show()

Test có JPEG cực thấp (thấy rõ khối 8×8), nhiễu Gauss rất mạnh, mờ, lệch màu, xoay/lật, nên augmentation phải đủ **mạnh**.

## 3. Augmentation mô phỏng biến đổi của test

In [ ]:
_ROT = [None, Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270]


def dihedral(im, k):
    """8 phép xoay/lật của hình vuông (k = 0..7)."""
    if k & 4: im = im.transpose(Image.FLIP_LEFT_RIGHT)
    return im if _ROT[k & 3] is None else im.transpose(_ROT[k & 3])


def jpeg(im, q):
    b = io.BytesIO(); im.save(b, 'JPEG', quality=int(q)); b.seek(0)
    return Image.open(b).convert('RGB')


jitter = T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.05)


def corrupt(im):
    im = dihedral(im, random.randrange(8))
    if random.random() < 0.6: im = jitter(im)
    if random.random() < 0.35: im = im.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.8)))
    if random.random() < 0.35:
        a = np.asarray(im, np.float32) + np.random.randn(*im.size[::-1], 3) * random.uniform(5, 60)
        im = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if random.random() < 0.45: im = jpeg(im, random.uniform(3, 40))
    return im


# Bản val "khó" cố định (seed cố định) để đo độ bền - chỉ số này mới giống test
seed_all(0)
Xva_hard = [corrupt(im) for im in Xva]

fig, ax = plt.subplots(1, 8, figsize=(16, 2.2))
for k in range(8): ax[k].imshow(Xva_hard[k]); ax[k].axis('off'); ax[k].set_title(classes[yva[k]], fontsize=8)
plt.show()


def f1(y, p): return round(f1_score(y, p, average='macro'), 4)


def write_sub(proba, tag):
    sub = pd.DataFrame({'id': test_ids, 'label': [classes[i] for i in proba.argmax(1)]})
    assert len(sub) == len(Xte) and sub.label.isin(classes).all()
    sub.to_csv('submission.csv', index=False); sub.to_csv(f'submission_{tag}.csv', index=False)
    log(f'đã ghi submission.csv (= submission_{tag}.csv)')

## 4. TẦNG 1 · Đặc trưng thủ công + LogisticRegression / XGBoost (CPU, không cần mạng nơ-ron)
Chọn đặc trưng **bất biến khi xoay/lật** (thống kê màu, histogram, LBP `uniform`, GLCM trung bình mọi hướng) để chịu được biến đổi của test;
mỗi ảnh train được làm hỏng thêm `N_AUG` lần để mô hình quen với nhiễu/JPEG/mờ.

In [ ]:
N_AUG = 3


def hand_feats(im):
    a = np.asarray(im, np.float32) / 255.0
    hsv = np.asarray(im.convert('HSV'), np.float32) / 255.0
    g = np.asarray(im.convert('L'))
    f = []
    for ch in np.concatenate([a, hsv], 2).transpose(2, 0, 1):                 # 6 kênh: R, G, B, H, S, V
        f += [ch.mean(), ch.std(), *np.percentile(ch, [10, 50, 90])]
    for c in range(3):
        f += list(np.histogram(a[..., c], bins=16, range=(0, 1))[0] / a[..., c].size)
    for P, R in [(8, 1), (16, 2)]:                                                # LBP uniform: bất biến khi xoay
        lbp = local_binary_pattern(g, P, R, method='uniform')
        f += list(np.histogram(lbp, bins=P + 2, range=(0, P + 2))[0] / lbp.size)
    glcm = graycomatrix(g // 8, distances=[1, 2], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=32, symmetric=True, normed=True)
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        f += list(graycoprops(glcm, prop).mean(1))                               # trung bình mọi hướng
    gy, gx = np.gradient(g.astype(np.float32))
    mag = np.hypot(gx, gy); f += [mag.mean(), mag.std(), np.percentile(mag, 90)]
    return np.array(f, np.float32)


feats = lambda ims, desc: np.stack([hand_feats(im) for im in tqdm(ims, desc=desc, leave=False)])
seed_all()
aug = lambda ims: [corrupt(im) for _ in range(N_AUG) for im in ims]
H_tr = feats(Xtr + aug(Xtr), 'train+aug'); h_tr = np.tile(ytr, N_AUG + 1)
H_va, H_va_hard = feats(Xva, 'val'), feats(Xva_hard, 'val hard')
log('đặc trưng thủ công:', H_tr.shape)

cands = {'LR': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000)),
         'XGB': XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', random_state=SEED, n_jobs=-1)}
res1 = {}
for name, clf in cands.items():
    clf.fit(H_tr, h_tr)
    res1[name] = {'clean': f1(yva, clf.predict(H_va)), 'hard': f1(yva, clf.predict(H_va_hard))}
    log(f'TẦNG 1 {name} val macro-F1:', res1[name])
T1 = max(res1, key=lambda k: res1[k]['hard'])
P1_va_hard = cands[T1].predict_proba(H_va_hard)
score_t1 = res1[T1]['hard']

Bản nộp tầng 1: fit lại mô hình tốt hơn trên train + val (đều có augmentation). Với test, lấy trung bình xác suất qua 2 phép xoay/lật.

In [ ]:
clf1 = cands[T1]
clf1.fit(np.concatenate([H_tr, feats(Xva + aug(Xva), 'val+aug')]), np.concatenate([h_tr, np.tile(yva, N_AUG + 1)]))
P1_te = (clf1.predict_proba(feats(Xte, 'test')) + clf1.predict_proba(feats([dihedral(im, 5) for im in Xte], 'test xoay'))) / 2
write_sub(P1_te, 'tier1')

## 5. TẦNG 2a · ResNet-18 khởi tạo ngẫu nhiên, train từ đầu (trong ngân sách thời gian)
Ảnh chỉ 64×64 nên đổi lớp đầu thành conv 3×3 stride 1 (giữ maxpool) cho bản đồ đặc trưng không bị thu nhỏ quá sớm.
Đo tốc độ thật vài bước, tự tính số epoch vừa ngân sách, rồi in ETA mỗi epoch. Chạy fp32 (fp16 trên GTX 16xx ra NaN).

In [ ]:
MEAN = np.stack([np.asarray(im, np.float32).mean((0, 1)) for im in Xtr]).mean(0) / 255
STD = np.stack([np.asarray(im, np.float32).std((0, 1)) for im in Xtr]).mean(0) / 255
post = T.Compose([T.ToTensor(), T.Normalize(MEAN.tolist(), STD.tolist())])
crop = T.RandomResizedCrop(64, scale=(0.7, 1.0), ratio=(0.9, 1.1))


class DS(torch.utils.data.Dataset):
    def __init__(self, ims, ys):
        self.ims, self.ys = ims, ys
    def __len__(self): return len(self.ims)
    def __getitem__(self, i): return post(crop(corrupt(self.ims[i]))), int(self.ys[i])


def make_model():
    m = torchvision.models.resnet18(weights=None, num_classes=len(classes))   # chỉ kiến trúc, trọng số ngẫu nhiên
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    return m.to(DEV, memory_format=torch.channels_last)


@torch.no_grad()
def run(m, ims, tta=True, bs=512):
    """Xác suất, trung bình qua 8 phép xoay/lật nếu tta."""
    m.eval(); ks = range(8) if tta else [0]; out = []
    for i in range(0, len(ims), bs):
        p = 0
        for k in ks:
            x = torch.stack([post(dihedral(im, k)) for im in ims[i:i + bs]]).to(DEV, memory_format=torch.channels_last)
            p = p + F.softmax(m(x), 1)
        out.append((p / len(ks)).cpu())
    return torch.cat(out).numpy()


def train(ims, ys, budget_min, lr=2e-3, max_epochs=200):
    seed_all()
    m = make_model()
    dl = torch.utils.data.DataLoader(DS(ims, ys), batch_size=BS, shuffle=True, num_workers=0, drop_last=True)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=5e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    def step(x, y, update=True):
        x, y = x.to(DEV, memory_format=torch.channels_last), y.to(DEV)
        loss = crit(m(x), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        if update: opt.step(); sched.step()
        return loss.item()

    # đo tốc độ thật bằng 4 bước không cập nhật trọng số
    m.train(); it = iter(dl); step(*next(it), update=False)
    t0 = time.time()
    for _ in range(3): step(*next(it), update=False)
    if DEV == 'cuda': torch.cuda.synchronize()
    sec_step = (time.time() - t0) / 3
    epochs = int(np.clip(budget_min * 60 / (sec_step * len(dl)), 2, max_epochs))
    log(f'{sec_step:.2f} s/bước × {len(dl)} bước/epoch, chạy {epochs} epoch ≈ {epochs * len(dl) * sec_step / 60:.1f} phút')

    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.2)
    t_train = time.time()
    for ep in range(epochs):
        m.train(); tot = 0.0
        for x, y in dl: tot += step(x, y)
        assert np.isfinite(tot), 'loss NaN/inf'
        if (ep + 1) % max(1, epochs // 10) == 0 or ep + 1 == epochs:
            done = (time.time() - t_train) / (ep + 1)
            log(f'  epoch {ep + 1}/{epochs} loss {tot / len(dl):.4f} · còn ~{done * (epochs - ep - 1) / 60:.1f} phút')
    return m


m1 = train(Xtr, ytr, BUDGET_FT)
P2_va_hard = run(m1, Xva_hard)
score_t2 = f1(yva, P2_va_hard.argmax(1))
score_ens = f1(yva, (P2_va_hard + P1_va_hard).argmax(1))
log('TẦNG 2a val macro-F1:', {'clean': f1(yva, run(m1, Xva).argmax(1)), 'hard': score_t2, 'hard (m1 + tầng 1)': score_ens})

In [ ]:
pred = (P2_va_hard + P1_va_hard).argmax(1) if score_ens > score_t2 else P2_va_hard.argmax(1)
plt.figure(figsize=(8, 6.5))
sns.heatmap(confusion_matrix(yva, pred), annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, cbar=False)
plt.title('tầng 2a trên val "hard"'); plt.xlabel('dự đoán'); plt.ylabel('nhãn'); plt.tight_layout(); plt.show()

P2_te = run(m1, Xte)
USE_T1 = score_ens > score_t2      # có cộng xác suất tầng 1 vào không (chọn trên val hard)
if max(score_t2, score_ens) > score_t1:
    write_sub(P2_te + P1_te if USE_T1 else P2_te, 'tier2a')
else:
    log('tầng 2a không tốt hơn tầng 1 trên val hard, nên giữ bản nộp tầng 1')

## 6. TẦNG 2b · Pseudo-label `unlabeled`, rồi train lại trên train + val + pseudo
Chỉ dùng ảnh `unlabeled` (dữ liệu được cấp để huấn luyện); **không** pseudo-label ảnh test vì luật cấm dùng test để huấn luyện.
Giữ ảnh có xác suất (TTA) ≥ `THR`, tối đa `CAP` ảnh/lớp để không lệch lớp. Không còn val sạch để đo, nên chỉ dùng để ensemble.

In [ ]:
THR, CAP = 0.85, 300
p_unl = run(m1, Xunl)
if USE_T1:
    p_unl = p_unl + clf1.predict_proba(feats(Xunl, 'unlabeled'))
p_unl = p_unl / p_unl.sum(1, keepdims=True)
conf, lab = p_unl.max(1), p_unl.argmax(1)
keep = []
for c in range(len(classes)):
    idx = np.where((lab == c) & (conf >= THR))[0]
    keep += list(idx[np.argsort(-conf[idx])][:CAP])
keep = np.array(sorted(keep))
X_ps, y_ps = [Xunl[i] for i in keep], lab[keep]
log(f'giữ {len(keep)}/{len(Xunl)} ảnh pseudo-label')

m2 = train(Xtr + Xva + X_ps, np.concatenate([ytr, yva, y_ps]), BUDGET_PS)
P3_te = run(m2, Xte)
final = P2_te + P3_te + (P1_te if USE_T1 else 0)
write_sub(final, 'tier2b')
torch.save({'m1': m1.state_dict(), 'm2': m2.state_dict()}, 'eurosat_resnet18_scratch.pt')   
pd.Series([classes[i] for i in final.argmax(1)]).value_counts()